# 01 — Exploratory data analysis: India's export footprint, 2019–2024

## Business question
What does India's export footprint for handicrafts and guar gum look like across 2019–2024 — by destination country, by product category, and by season — and what does that tell us about the Jodhpur export cluster?

## Data used
- **Source:** UN Comtrade Plus, India as reporter (code 699), monthly grain
- **HS codes (per PRD §8.1):** 440929 (wood non-coniferous), 442090 (wooden articles), 330749 (room fragrance), 130232 (guar gum), 130239 (other plant mucilages)
- **Time range:** January 2019 – December 2024 (5 full years + partial 2024)
- **Records:** ~12,800 monthly country-aggregated rows after cleaning
- **Storage:** Supabase Postgres `fact_shipment` (this notebook reads directly via SQLAlchemy; falls back to local parquet if the DB is paused)

## Key assumption
India-level Comtrade data is treated as a reasonable proxy for Jodhpur cluster trends:
- **Guar gum:** ~95 % of India's commercial guar processing is in Rajasthan/Haryana, with Jodhpur a major hub (per IIM-A Hindustan Gum case)
- **Wooden articles (HS 442090):** Jodhpur + Moradabad together dominate India's export of this code

We cannot disaggregate to *Jodhpur-only* shipments from public Comtrade data. That limitation is documented and surfaced again in the findings cell below.

In [ ]:
# Imports + plot defaults
from __future__ import annotations

import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

warnings.filterwarnings('ignore', category=UserWarning)
load_dotenv(Path('..') / '.env')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.0)
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.titleweight'] = 'bold'

In [ ]:
# Load data — Postgres first (canonical), parquet as fallback
def load_exports() -> pd.DataFrame:
    """Pull the canonical export dataset from Supabase, falling back to the
    local Parquet artifact if the DB is paused / offline. Either path yields
    the same schema — we explicitly normalise column names so downstream
    cells don't care which source was used."""
    parquet_path = Path('..') / 'data' / 'processed' / 'exports_clean.parquet'
    db_url = os.getenv('DATABASE_URL')
    if db_url:
        try:
            engine = create_engine(db_url, pool_pre_ping=True)
            sql = text("""
                SELECT  f.shipment_date,
                        EXTRACT(YEAR  FROM f.shipment_date)::int  AS year,
                        EXTRACT(MONTH FROM f.shipment_date)::int  AS month,
                        dp.hs_code,
                        dp.hs_desc,
                        dp.category                              AS hs_category,
                        dc.iso_alpha3                            AS dest_iso_alpha3,
                        dc.country_name                          AS dest_country_name,
                        f.fob_usd,
                        f.quantity_kg,
                        f.unit_price_usd,
                        f.is_outlier
                FROM    fact_shipment f
                JOIN    dim_country   dc ON dc.country_id = f.dest_country_id
                JOIN    dim_product   dp ON dp.product_id = f.product_id
            """)
            df = pd.read_sql(sql, engine, parse_dates=['shipment_date'])
            print(f'Loaded {len(df):,} rows from Supabase Postgres')
            return df
        except Exception as exc:
            print(f'Postgres unavailable ({exc.__class__.__name__}); falling back to parquet')
    df = pd.read_parquet(parquet_path)
    df['hs_category'] = df['hs_code'].map({
        '440929': 'Handicraft-Wood',
        '442090': 'Handicraft-Wood',
        '330749': 'Handicraft-Fragrance',
        '130232': 'Guar-Refined',
        '130239': 'Guar-Other',
    })
    print(f'Loaded {len(df):,} rows from {parquet_path.name}')
    return df

df = load_exports()
df.head()

In [ ]:
# Sanity check — basic dataset shape
print(f'Rows:                  {len(df):>10,}')
print(f'Unique destinations:   {df["dest_iso_alpha3"].nunique():>10,}')
print(f'Unique HS codes:       {df["hs_code"].nunique():>10,}')
print(f'Date range:            {df["shipment_date"].min():%Y-%m-%d} → {df["shipment_date"].max():%Y-%m-%d}')
print(f'Cumulative FOB (USD):  {df["fob_usd"].sum()/1e9:>10.2f} B')
print(f'Cumulative quantity:   {df["quantity_kg"].sum()/1e9:>10.2f} B kg')
print(f'Outliers flagged:      {df["is_outlier"].sum():>10,}  ({df["is_outlier"].mean()*100:.2f}%)')

## Chart 1 — Total monthly exports across 2019–2024

The headline shape: does demand trend up, down, or flat? Are there visible seasonal swings? Pre-Christmas months (Sep–Nov) are shaded — the PRD's claim is that they spike.

In [ ]:
monthly = df.groupby('shipment_date', as_index=False)['fob_usd'].sum()
monthly['fob_usd_m'] = monthly['fob_usd'] / 1e6

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly['shipment_date'], monthly['fob_usd_m'], linewidth=1.8, color='#264653')
ax.fill_between(monthly['shipment_date'], monthly['fob_usd_m'], alpha=0.18, color='#264653')

# Highlight Sep–Nov peak windows for every year
for yr in sorted(df['shipment_date'].dt.year.unique()):
    ax.axvspan(pd.Timestamp(yr, 9, 1), pd.Timestamp(yr, 11, 30), alpha=0.07, color='#e76f51')

ax.set_title('India total monthly exports — 5 HS codes combined (2019–2024)')
ax.set_xlabel('Month')
ax.set_ylabel('FOB exports (USD millions)')
ax.text(0.01, 0.96, 'Shaded bands = Sep–Nov pre-Christmas window',
        transform=ax.transAxes, fontsize=9, color='#e76f51', verticalalignment='top')
plt.tight_layout()
plt.show()

peak_avg = monthly[monthly['shipment_date'].dt.month.isin([9, 10, 11])]['fob_usd_m'].mean()
annual_avg = monthly['fob_usd_m'].mean()
print(f'\nPeak-month average (Sep–Nov): ${peak_avg:.1f} M')
print(f'Annual monthly average:       ${annual_avg:.1f} M')
print(f'Peak vs annual:               {(peak_avg/annual_avg - 1)*100:+.1f}%')

## Chart 2 — Top 10 destinations by cumulative FOB

Where does India's handicraft+guar export value actually go? Concentration risk is one of the PRD's five pain points — let's see what it looks like.

In [ ]:
top10 = (
    df.groupby('dest_country_name')['fob_usd'].sum()
      .sort_values(ascending=True)
      .tail(10) / 1e6
)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top10.index, top10.values, color='#2a9d8f')
for bar, val in zip(bars, top10.values):
    ax.text(val * 1.01, bar.get_y() + bar.get_height()/2,
            f'${val:,.0f}M', va='center', fontsize=10)
ax.set_xlabel('Cumulative exports 2019–2024 (USD millions)')
ax.set_title('Top 10 destination countries — all 5 HS codes combined')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

total = df['fob_usd'].sum() / 1e6
top10_share = top10.sum() / total * 100
top1_share = top10.iloc[-1] / total * 100
print(f'\nTop 10 destinations = {top10_share:.1f}% of total cumulative FOB')
print(f'Top 1 destination ({top10.index[-1]}) alone = {top1_share:.1f}% of total')

## Chart 3 — HS code mix year over year

Which products drive total value, and how has the mix shifted? Guar (HS 130232/130239) tends to dominate by volume; handicrafts dominate by margin. Stacked area shows the value share.

In [ ]:
mix = (df.groupby(['year', 'hs_code'])['fob_usd'].sum()
         .unstack(fill_value=0) / 1e6)

fig, ax = plt.subplots(figsize=(12, 5))
mix.plot(kind='area', stacked=True, ax=ax, alpha=0.85, linewidth=0.5,
         color=['#264653', '#2a9d8f', '#e9c46a', '#f4a261', '#e76f51'])
ax.set_title('Export value composition by HS code, year over year')
ax.set_xlabel('Year')
ax.set_ylabel('FOB exports (USD millions)')
ax.legend(title='HS code', loc='center left', bbox_to_anchor=(1.01, 0.5), frameon=False)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

category_mix = df.groupby('hs_category')['fob_usd'].sum() / df['fob_usd'].sum() * 100
print('\nValue share by product category:')
for cat, pct in category_mix.sort_values(ascending=False).items():
    print(f'  {cat:<25} {pct:5.1f}%')

## Chart 4 — Seasonal heatmap (year × month)

If the seasonal peak is real, this heatmap should glow red in Sep–Nov rows. If it's a myth, the heatmap will look uniform.

In [ ]:
season = (df.groupby(['year', 'month'])['fob_usd'].sum()
            .unstack(fill_value=0) / 1e6)

fig, ax = plt.subplots(figsize=(13, 5))
sns.heatmap(season, annot=True, fmt='.0f', cmap='YlOrRd',
            cbar_kws={'label': 'FOB exports (USD millions)'}, ax=ax,
            annot_kws={'fontsize': 9})
ax.set_title('Monthly export value heatmap — annotations in USD millions')
ax.set_xlabel('Month')
ax.set_ylabel('Year')
plt.tight_layout()
plt.show()

# Year-by-year peak premium
print('\nPeak season vs annual average, year by year:')
for yr in season.index:
    row = season.loc[yr]
    if row.sum() == 0:
        continue
    peak = row.reindex([9, 10, 11]).mean()
    annual = row[row > 0].mean()
    if pd.notna(peak) and pd.notna(annual) and annual > 0:
        pct = (peak / annual - 1) * 100
        print(f'  {yr}: peak-month avg ${peak:6.1f} M vs annual ${annual:6.1f} M  ({pct:+.1f}%)')

## Chart 5 — Unit price distribution per HS code

The five HS codes span very different price regimes. Wooden articles sell at single-digit USD/kg; perfuming preparations can be 100×. Clipping at the 95th percentile per HS to keep the chart legible (the `is_outlier` flag survives in the full dataset).

In [ ]:
hs_codes = sorted(df['hs_code'].unique())
fig, axes = plt.subplots(1, len(hs_codes), figsize=(18, 4), sharey=False)

for ax, hs in zip(axes, hs_codes):
    sub = df[df['hs_code'] == hs]['unit_price_usd']
    capped = sub.clip(upper=sub.quantile(0.95))
    ax.hist(capped, bins=30, color='#2a9d8f', edgecolor='#264653', alpha=0.85)
    ax.axvline(sub.median(), color='#e76f51', linestyle='--', linewidth=1.5,
               label=f'median ${sub.median():.2f}/kg')
    ax.set_title(f'HS {hs}')
    ax.set_xlabel('USD per kg')
    ax.legend(fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)

fig.suptitle('Unit price distribution per HS code (clipped at p95 for legibility)', y=1.02)
plt.tight_layout()
plt.show()

print('\nMedian unit price (USD/kg) by HS code:')
for hs in hs_codes:
    sub = df[df['hs_code'] == hs]['unit_price_usd']
    print(f'  HS {hs}: median ${sub.median():>6.2f}/kg   p95 ${sub.quantile(0.95):>7.2f}/kg')

## Chart 6 — Year-over-year growth heatmap, top 15 destinations

Where is demand growing or shrinking, and how fast? Green cells = growth, red = contraction. This is the raw material the K-means segmentation in notebook 03 will use to label countries as Rising / Core / Declining.

In [ ]:
top15_names = (df.groupby('dest_country_name')['fob_usd'].sum()
                 .sort_values(ascending=False).head(15).index.tolist())

yearly_top15 = (df[df['dest_country_name'].isin(top15_names)]
                  .groupby(['dest_country_name', 'year'])['fob_usd'].sum()
                  .unstack(fill_value=0) / 1e6)
yoy = yearly_top15.pct_change(axis=1) * 100
yoy = yoy.iloc[:, 1:]
yoy = yoy.reindex(top15_names)

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(yoy, annot=True, fmt='.0f', cmap='RdYlGn', center=0,
            cbar_kws={'label': 'YoY % change'}, ax=ax, vmin=-100, vmax=200,
            annot_kws={'fontsize': 9})
ax.set_title('Year-over-year growth — top 15 destinations')
ax.set_xlabel('Year')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

## Findings

*(Numbers below are the verified outputs of this run on the 12,828-row cleaned dataset, 2019–2024.)*

1. **The "cluster-wide Sep–Nov peak" the industry plans around does not exist.** Aggregated across all 5 HS codes, the Sep–Nov window averages **−8.0% *below*** the all-year monthly mean — it is a trough, not a peak. The genuine pre-Christmas handicraft uplift (+9.1% for HS 440929/442090/330749) is real but is swamped by guar gum's counter-cyclical, fracking-driven demand (−11.1% in Sep–Nov). Notebook 02 quantifies and models this. Operational consequence: one cluster-wide production calendar destroys value — handicraft and guar need separate plans.

2. **Concentration risk is severe.** The top 10 destinations account for **76%** of cumulative FOB; the single largest — **USA — is 34%** ($867M of $2.52B). China (11%) and Germany (11%) follow, then a steep drop-off. A contraction in any one of the top three (as guar did during the 2024 US fracking slowdown) swings cluster revenue sharply. This is the operational form of PRD pain point 4; it is sized in rupee terms in notebook 05.

3. **Guar gum is the value engine — and the single point of failure.** HS 130232 alone is **83%** of cumulative export value (guar incl. 130239 ≈ 85%); the three handicraft codes together are ~15%. Handicrafts run at far higher per-kg unit prices but tiny absolute volume. The strategic question for the cluster is therefore *defending* guar volume (notebook 06's rig-count-driven forecast) while *growing* the higher-margin handicraft mix.

4. **Several top markets swing violently year-over-year.** Multiple top-15 destinations flip between large positive and negative YoY moves (Chart 6) — these are exactly the markets the segmentation notebook (03) flags, and the ones whose erode-while-still-large profile makes them the "Core-but-Declining" watchlist.

## Limitations

- **India-level, not Jodhpur-level.** Comtrade aggregates by country. We treat India-origin guar and India-origin HS 442090/330749 as proxies for Jodhpur cluster trends, but cannot isolate Jodhpur shipments from this data.
- **No buyer-level visibility.** Comtrade rows are monthly country aggregates — no individual buyer/exporter names. Notebook 05 works around this with a documented country-as-buyer proxy.
- **Sparse late-2024 data.** Comtrade reporters submit with a 6–18 month lag; 2024 figures will revise upward as later submissions land. Every notebook stamps a provenance banner with the data vintage.
- **"Other Asia, nes" and similar aggregates** were dropped during cleaning — we chose to lose ~1.5% of rows rather than introduce phantom-country distortion.

## Recommendation

Three operational moves a Jodhpur exporter could make on the basis of *this single notebook*:

1. **Run two production calendars, not one.** The cluster-level Sep–Nov "peak" is actually −8.0% (Finding 1). Handicraft producers should ramp ~6–8 weeks ahead of Sep–Nov for the genuine +9.1% pre-Christmas window; guar producers should ignore that heuristic entirely and track US rig counts (notebook 06). Treating the cluster as one seasonal line destroys signal.
2. **Audit dependency on the top-3 destinations.** Concentration is 76% in the top 10 and 34% in the USA alone. Design at least one back-up channel per headline market — typically a Rising-Stars country surfaced in notebook 03.
3. **Treat guar volume defence and handicraft margin growth as separate strategic motions.** Their seasonality, buyer profile, and price elasticity differ; an 85%-of-revenue commodity and a high-margin craft line do not share a strategy.

## Next notebook

`02_demand_forecast.ipynb` — formalises the seasonality finding above with a `statsmodels` SARIMAX model (Prophet was dropped — its CmdStan build is unavailable in this environment; see `docs/ARCHITECTURE.md` §4.1), validated by rolling-origin cross-validation against the PRD's MAPE target.